In [ ]:
import os
# 禁止 JAX 预占所有显存，用多少拿多少
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import pennylane as qml
import jax
import jax.numpy as jnp
import numpy as np
import scipy.sparse as sp
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
import optax
import catalyst
from catalyst import qjit
import time


qml.about()

In [2]:
import jax
print('✅ JAX version:', jax.devices())
import jax
print('✅ JAX version:', jax.__version__)
print('✅ Devices:', jax.devices())
if any(d.platform == 'gpu' for d in jax.devices()):
    print('🎉 GPU is working!')
else:
    print('⚠️ No GPU detected')

✅ JAX version: [CudaDevice(id=0)]
✅ JAX version: 0.6.2
✅ Devices: [CudaDevice(id=0)]
🎉 GPU is working!


In [3]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=6 N=4.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_3 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_3.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_3.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_3.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_3.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=3, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

✅ 矩阵调用成功！
矩阵格式：csr
矩阵形状：(11424, 11424)
非零元素个数：548306
最小特征值: [-28.08061951 -11.19353886   8.38332622]
-28.08061951456625


In [4]:
import pennylane as qml
import jax
import jax.numpy as jnp
import numpy as np  # 用于预处理，非计算图部分
import time
import optax  # JAX 的标准优化库 (需要 pip install optax)
from catalyst import qjit, grad

# =================== 0. 辅助函数 (保持不变) ===================
def get_Hami(H):
    # 安全地将 H 转换为无梯度的 NumPy 数组
    if hasattr(H, 'toarray'):
        H = H.toarray()
    if hasattr(H, 'detach'):
        H = H.detach().cpu().numpy()
    elif hasattr(H, 'numpy'):
        H = H.numpy()
    else:
        H = np.asarray(H)

    H_dense = np.array(H, copy=True)
    d = H_dense.shape[0]
    Nq = int(np.ceil(np.log2(d)))
    l = 2 ** Nq

    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    np.fill_diagonal(Hami, 1000)
    Hami[:d, :d] = H_dense

    return Hami, Nq

H_sy,n_qubits= get_Hami(H)



In [5]:
import numpy as np
from math import log2
import numpy as np
from qiskit.quantum_info import Operator, SparsePauliOp


def gray_order(n: int):
    """
    生成 n 比特 Gray code 的顺序（对应的十进制索引列表）
    例如 n=2 时返回 [0, 1, 3, 2]
    """
    return [i ^ (i >> 1) for i in range(2 ** n)]

def H_gray(H, n: int | None = None):

    H = np.asarray(H)

    dim = H.shape[0]

    # 自动推断 n
    if n is None:
        n_est = log2(dim)
        if abs(round(n_est) - n_est) > 1e-9:
            raise ValueError("无法从矩阵维度推断出整数量子比特。")
        n = int(round(n_est))

    if dim != 2 ** n:
        raise ValueError(f"矩阵维度 {dim} 与 n={n} 不匹配（应为 2^n={2**n}）。")

    # 获取 Gray code 顺序对应的索引
    idx = np.array(gray_order(n))

    # 等价于 P^T H P：行和列都按同一个置换重排
    H_gray = H[np.ix_(idx, idx)]
    return H_gray

H_gray = H_gray(H_sy)

import scipy.sparse as sp

# 1. 既然你有 H_gray (稠密矩阵)，直接转为 scipy 稀疏矩阵
# (这一步是瞬间完成的)
H_sparse = sp.csr_matrix(H_gray)





In [6]:
# =================== 5. 电路定义 (GPU Device) ===================
depth = 200 # 演示用，深度可调
steps = 5000 # 演示用


dev = qml.device("lightning.gpu", wires=n_qubits)


@qml.qnode(dev, diff_method="adjoint")
def cost(params):
    # 初始化 HF 态 (全0态作为示例)
    hf = jnp.zeros(n_qubits, dtype=int)
    qml.BasisState(hf, wires=range(n_qubits))

    for i in range(n_qubits):
        qml.Hadamard(wires=i)

    for d in range(depth):
        for i in range(n_qubits):
            qml.RY(params[d*n_qubits+i], wires=i)

        # 偶数索引比特 → 下一个比特（j=0,2,4...）
        for j in range(0, n_qubits-1, 2):
            qml.CNOT(wires=[j, (j+1) % n_qubits])
        # 奇数索引比特 → 下一个比特（j=1,3,5...）
        for j in range(1, n_qubits-1, 2):
            qml.CNOT(wires=[j, (j+1) % n_qubits])
    return qml.expval(qml.SparseHamiltonian(H_sparse, wires=range(n_qubits)))

In [7]:
# 4. 定义优化器
# 使用 optax，但我们在纯 Python 模式下运行它

n_params = depth * n_qubits
init_params = jnp.zeros(n_params)

optimizer = optax.adam(learning_rate=0.01)
opt_state = optimizer.init(init_params)
params = init_params

# 5. 训练循环 (纯 Python, 无 @jax.jit)
print("开始训练 (Adjoint Differentiation mode)...")
import time
start_total = time.time()

for i in range(steps):
    step_start = time.time()

    # 计算梯度和能量
    # 这一步调用 C++ 后端，速度很快
    value, grads = jax.value_and_grad(cost)(params)

    # 更新参数
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    # 打印进度
    if i % 10 == 0:
        print(f"Step {i:3d} | Energy: {value:.8f} Ha | Time: {time.time()-step_start:.4f}s")

print(f"训练完成，总耗时: {time.time()-start_total:.2f}s")

开始训练 (Adjoint Differentiation mode)...


XlaRuntimeError: RESOURCE_EXHAUSTED: /tmp/tempfile-ps-952012f48c074bcb-3716062-64594a4347826; Disk quota exceeded
	Unable to write PTX contents to: /tmp/tempfile-ps-952012f48c074bcb-3716062-64594a4347826